# Librerias

In [1]:
import os
import sys
import json
import glob
import zipfile
import hashlib
import importlib

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import geopandas as gpd
import requests

from pyproj import Geod, Transformer
from shapely import STRtree
from shapely.validation import make_valid


def localizar_constantes(inicio=None):
    """Busca constantes.py subiendo desde la carpeta del cuaderno.

    Misma estrategia que B1: ninguna ruta absoluta queda escrita, de modo
    que el cuaderno corre en cualquier máquina donde exista el árbol del
    proyecto.
    """
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in (inicio, *inicio.parents):
        for candidato in (carpeta / "constantes.py",
                          carpeta / "Bloque A" / "constantes.py"):
            if candidato.is_file():
                return candidato.parent
    raise FileNotFoundError(
        f"No se encontró constantes.py subiendo desde {inicio}"
    )


RUTA_CONSTANTES = localizar_constantes()
if str(RUTA_CONSTANTES) not in sys.path:
    sys.path.insert(0, str(RUTA_CONSTANTES))

import constantes
constantes = importlib.reload(constantes)

print("constantes.py  :", RUTA_CONSTANTES / "constantes.py")
print("RUTA_PROYECTO  :", constantes.RUTA_PROYECTO)
print("geopandas      :", gpd.__version__)

constantes.py  : F:\Tesis\Bloque A\constantes.py
RUTA_PROYECTO  : F:\Tesis
geopandas      : 1.1.4


# Descarga e inspeccion

In [2]:
# BLOQUE B2 — Reproyección verificada y justificación empírica de umbrales
#
# ORDEN DEL BLOQUE B (lineal, sin ciclos):
#   B1    clasifica con OSM y OCM; deja 6 casos sin verificar
#   B2    mide la escala urbana, justifica los umbrales y resuelve esos casos
#   B2.1  aplica los veredictos y documenta el contraste contra B1
#
# B2 no depende de variables en memoria de B1 ni modifica sus salidas.
#
# FUENTE NUEVA: Manzana. Bogotá D.C. — Unidad Administrativa Especial de
# Catastro Distrital (UAECD), integrada al Mapa de Referencia de IDECA.
#   Licencia:   Creative Commons Attribution 4.0
#   Resolución: 1:1000
#   CRS ficha:  EPSG:4686 (MAGNA-SIRGAS geographic 2D)
#   Versión:    07.26 (fecha del dato 2026-07-26, actualización trimestral)
#   Ficha:      https://datosabiertos.bogota.gov.co/dataset/manzana
#   Técnica:    https://www.ideca.gov.co/sites/default/files/CO_Manzana_MR.pdf
#
# Se usa la versión fechada del recurso, no un enlace "actual", para que la
# descarga sea reproducible.
# =============================================================================

RUTA_A2    = constantes.ruta_entregable("A2")
RUTA_A3    = constantes.ruta_entregable("A3")
RUTA_B1    = constantes.ruta_entregable("B1")
RUTA_OUT   = constantes.ruta_entregable("B2")
RUTA_CRUDO = RUTA_OUT / "crudo"
RUTA_OUT.mkdir(parents=True, exist_ok=True)
RUTA_CRUDO.mkdir(parents=True, exist_ok=True)

CRS_GEO  = constantes.CRS_WGS84     # EPSG:4326
CRS_PROY = constantes.CRS_METRICO   # EPSG:9377

# --- Umbrales heredados de B1. B2 los somete a análisis, no los redefine. ---
UMBRAL_CONFIRMADA  = constantes.UMBRAL_CONFIRMADA    #  25 m  — anclaje externo (GPS)
UMBRAL_PROBABLE    = constantes.UMBRAL_PROBABLE      # 100 m  — anclaje interno (manzana)
UMBRAL_TEXTUAL     = constantes.UMBRAL_TEXTUAL       # 0.45   — pendiente
RADIO_BUSQUEDA_MAX = constantes.RADIO_BUSQUEDA_MAX   # 500 m  — pendiente
RADIO_SECUNDARIO   = constantes.RADIO_SECUNDARIO     # 300 m  — pendiente
RADIO_DUPLICADO    = constantes.RADIO_DUPLICADO      #  50 m  — pendiente

# --- Parámetros propios de B2 -----------------------------------------------
# Se declaran con su criterio, no como números sueltos. Los tres primeros son
# filtros de la capa catastral, no umbrales de clasificación: no entran en
# ninguna decisión sobre estaciones, solo en la medición de la trama urbana.
# Aun así se someten a sensibilidad, porque B2 no puede exigirle a B1 lo que
# no se exige a sí mismo.

LIM_INF_M2    = 100.0       # descarta residuos geométricos (predio < 10 x 10 m)
LIM_SUP_M2    = 100_000.0   # descarta supermanzanas (>10 ha: parques, aeropuerto)
RADIO_LOCAL_M = 1000.0      # entorno de las estaciones para medir la trama

URL_MANZANA = ("https://datosabiertos.bogota.gov.co/dataset/"
               "21c8667c-3a46-454e-bf79-58c824cabf63/resource/"
               "69d42caf-3cd5-4336-935d-1ca9799527c0/download/manz.shp.07.26.zip")
VERSION_MANZANA = "07.26"

# Hash esperado del crudo. Se deja en None la primera vez; la celda de
# descarga lo registra y desde entonces cualquier cambio del archivo hace
# fallar el cuaderno en lugar de cambiar los resultados en silencio.
SHA256_MANZANA_ESPERADO = None

GEOD = Geod(ellps="GRS80")  # elipsoide de MAGNA-SIRGAS

# Entradas desde disco

In [3]:
# ---------------------------------------------------------------------------
# Procedencia: SHA-256 de todo lo que B2 consume, encadenado a los crudos
# congelados que A2 y A3 registraron. Si un insumo se regenera, el cuaderno
# falla ruidosamente en vez de producir otros números sin avisar.
# ---------------------------------------------------------------------------

def sha256_archivo(ruta, bloque=1 << 20):
    h = hashlib.sha256()
    with open(ruta, "rb") as fh:
        for trozo in iter(lambda: fh.read(bloque), b""):
            h.update(trozo)
    return h.hexdigest()


INSUMOS = [
    RUTA_B1 / "B1_pares_primarios.csv",
    RUTA_B1 / "B1_clasificacion_cuatro_categorias.csv",
    RUTA_A2 / "A2_estaciones_OSM_Bogota.geojson",
    RUTA_A2 / "A2_estaciones_OSM_Medellin.geojson",
    RUTA_A3 / "A3_OpenChargeMap_Bogota_estaciones.csv",
    RUTA_A3 / "A3_OpenChargeMap_Medellin_estaciones.csv",
]

faltantes = [r for r in INSUMOS if not r.is_file()]
if faltantes:
    raise FileNotFoundError(
        "Faltan insumos:\n" + "\n".join(f"  {r}" for r in faltantes) +
        "\nCorre A2, A3 y B1 antes que B2."
    )

MANIFIESTO = {r.name: sha256_archivo(r) for r in INSUMOS}

# Encadenamiento con los crudos congelados: A2 y A3 dejaron sidecars .sha256
# junto a sus respuestas crudas. Se copian tal cual al manifiesto de B2.
for carpeta in (RUTA_A2, RUTA_A3):
    for sidecar in sorted(carpeta.glob("*.sha256")):
        MANIFIESTO[f"crudo::{sidecar.stem}"] = (
            sidecar.read_text(encoding="utf-8").split()[0]
        )

ARCH_MANIFIESTO = RUTA_OUT / "B2_manifiesto_insumos.json"
if ARCH_MANIFIESTO.is_file():
    previo = json.loads(ARCH_MANIFIESTO.read_text(encoding="utf-8"))
    cambiados = {k for k in set(previo) | set(MANIFIESTO)
                 if previo.get(k) != MANIFIESTO.get(k)}
    if cambiados:
        raise RuntimeError(
            "Los insumos cambiaron respecto de la corrida registrada: "
            f"{sorted(cambiados)}. Revisa qué se regeneró y, si el cambio es "
            f"intencional, borra {ARCH_MANIFIESTO.name} y vuelve a correr."
        )
    print("Procedencia verificada contra el manifiesto previo.")
else:
    ARCH_MANIFIESTO.write_text(
        json.dumps(MANIFIESTO, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print("Manifiesto de insumos creado.")

# ---------------------------------------------------------------------------
# Carga. Se usa el módulo compartido: los identificadores quedan como texto,
# que es lo que impide que '475409' y '475409.0' se traten como registros
# distintos.
# ---------------------------------------------------------------------------

osm, ocm, PARES, CASOS = constantes.cargar_resultados_b1()

OSM = osm.set_index("id")
OCM = ocm.set_index("id")

if not OSM.index.is_unique:
    raise ValueError("Hay osm_uid repetidos en el inventario de A2.")
if not OCM.index.is_unique:
    raise ValueError("Hay ocm_id repetidos en el inventario de A3.")

sin_ocm = set(PARES.ocm_id) - set(OCM.index)
sin_osm = set(PARES.osm_uid) - set(OSM.index)
if sin_ocm or sin_osm:
    raise ValueError(
        f"Pares de B1 que no existen en los inventarios. "
        f"OCM huérfanos: {sorted(sin_ocm)} | OSM huérfanos: {sorted(sin_osm)}"
    )

# Versiones proyectadas, una sola vez, reutilizadas en todo el cuaderno.
def proyectar(tabla):
    g = gpd.GeoDataFrame(
        tabla.copy(),
        geometry=gpd.points_from_xy(tabla.lon, tabla.lat),
        crs=CRS_GEO,
    ).to_crs(CRS_PROY)
    return g

OSM_P, OCM_P = proyectar(OSM), proyectar(OCM)

n_degr = int((PARES.categoria == "DEGRADADO").sum())
print(f"\nPares primarios: {len(PARES)}  (degradados: {n_degr})")
print(f"Casos ambiguos:  {len(CASOS)} -> {sorted(CASOS)}")
print(f"Estaciones:      OCM {len(OCM)} | OSM {len(OSM)}")

assert len(PARES) == 20, f"Se esperaban 20 pares primarios, hay {len(PARES)}"
assert n_degr == 3,      f"Se esperaban 3 degradados, hay {n_degr}"
assert len(CASOS) == 6,  f"Se esperaban 6 casos, hay {len(CASOS)}"

Procedencia verificada contra el manifiesto previo.

Pares primarios: 20  (degradados: 3)
Casos ambiguos:  6 -> ['C3-1', 'C3-2', 'C3-3', 'N1-1', 'N1-2', 'N1-3']
Estaciones:      OCM 38 | OSM 53


# Descarga con registro de procedencia

In [4]:
ZIP_LOCAL  = RUTA_CRUDO / f"manzana_{VERSION_MANZANA}.zip"
META_LOCAL = RUTA_CRUDO / f"manzana_{VERSION_MANZANA}.meta.json"

# Mismo patrón que A2/A3: crudo + sidecar de metadatos. Si el crudo existe,
# el cuaderno no toca la red y la fecha de descarga sale del sidecar, no del
# reloj. Así dos corridas del mismo archivo producen el mismo JSON de salida.

if ZIP_LOCAL.is_file() and META_LOCAL.is_file():
    procedencia = json.loads(META_LOCAL.read_text(encoding="utf-8"))
    print("Crudo ya congelado; no se consulta el servidor.")
else:
    print("Descargando…")
    respuesta = requests.get(
        URL_MANZANA, timeout=600, stream=True,
        headers={"User-Agent": constantes.USER_AGENT},
    )
    respuesta.raise_for_status()
    with open(ZIP_LOCAL, "wb") as fh:
        for trozo in respuesta.iter_content(chunk_size=1 << 20):
            fh.write(trozo)

    procedencia = {
        "fuente": "Manzana. Bogotá D.C. (UAECD / Mapa de Referencia IDECA)",
        "url": URL_MANZANA,
        "version": VERSION_MANZANA,
        "licencia": "CC BY 4.0",
        "resolucion": "1:1000",
        "crs_declarado_ficha": "EPSG:4686",
        "fecha_descarga": datetime.now(timezone.utc).isoformat(),
        "bytes": ZIP_LOCAL.stat().st_size,
        "sha256": sha256_archivo(ZIP_LOCAL),
    }
    META_LOCAL.write_text(
        json.dumps(procedencia, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (RUTA_CRUDO / f"manzana_{VERSION_MANZANA}.zip.sha256").write_text(
        procedencia["sha256"] + "\n", encoding="utf-8"
    )

# Integridad: el archivo en disco tiene que seguir siendo el que se registró.
sha_actual = sha256_archivo(ZIP_LOCAL)
if sha_actual != procedencia["sha256"]:
    raise RuntimeError(
        "El crudo de manzana no coincide con su hash registrado. "
        "Borra el zip, el .sha256 y el .meta.json para volver a descargar."
    )
if SHA256_MANZANA_ESPERADO and sha_actual != SHA256_MANZANA_ESPERADO:
    raise RuntimeError("El crudo no coincide con SHA256_MANZANA_ESPERADO.")

FECHA_DESCARGA_MANZANA = procedencia["fecha_descarga"]
for k, v in procedencia.items():
    print(f"  {k}: {v}")

DIR_SHP = RUTA_CRUDO / f"manzana_{VERSION_MANZANA}"
if not DIR_SHP.is_dir():
    with zipfile.ZipFile(ZIP_LOCAL) as z:
        z.extractall(DIR_SHP)

shps = sorted(DIR_SHP.rglob("*.shp"))
if len(shps) != 1:
    raise RuntimeError(f"Se esperaba un solo .shp, se hallaron {len(shps)}: {shps}")
ARCH_SHP = shps[0]
print("\nShapefile:", ARCH_SHP)

Descargando…
  fuente: Manzana. Bogotá D.C. (UAECD / Mapa de Referencia IDECA)
  url: https://datosabiertos.bogota.gov.co/dataset/21c8667c-3a46-454e-bf79-58c824cabf63/resource/69d42caf-3cd5-4336-935d-1ca9799527c0/download/manz.shp.07.26.zip
  version: 07.26
  licencia: CC BY 4.0
  resolucion: 1:1000
  crs_declarado_ficha: EPSG:4686
  fecha_descarga: 2026-09-15T03:40:30.781663+00:00
  bytes: 34374633
  sha256: 04be5ad368d471cdfc40093306c709c962ea0fd90d4e0a61a150aa0c24f5b6d3

Shapefile: F:\Tesis\Bloque B\Entregable B2\crudo\manzana_07.26\MANZ.shp


# Lectura, reparación y reproyección

In [5]:
mz = gpd.read_file(ARCH_SHP)
n_original = len(mz)
print("Registros:", n_original)
print("CRS del archivo:", mz.crs)
print("Tipos de geometría:", mz.geom_type.value_counts().to_dict())

# El .prj declara EPSG:4326 y la ficha declara EPSG:4686. La discrepancia se
# documenta y se cuantifica más adelante: entre ambos datums el
# desplazamiento en Colombia es milimétrico, muy por debajo de los umbrales.
procedencia["crs_declarado_archivo"] = str(mz.crs)

n_inv = int((~mz.is_valid).sum())
if n_inv:
    mz.loc[~mz.is_valid, "geometry"] = mz.loc[~mz.is_valid, "geometry"].apply(make_valid)

antes_tipo = len(mz)
mz = mz[mz.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
print(f"Geometrías inválidas reparadas: {n_inv}")
print(f"Descartadas por no ser polígono tras reparar: {antes_tipo - len(mz)}")
print(f"Inválidas restantes: {int((~mz.is_valid).sum())}")

# Reproyección única. MZ_P conserva TODAS las manzanas y se usa para la
# prueba de contención; MZ es el subconjunto urbano y se usa para medir la trama.
MZ_P = mz.to_crs(CRS_PROY)
MZ_P["area_m2"] = MZ_P.area
MZ_P["perim_m"] = MZ_P.length

n_bajo = int((MZ_P.area_m2 < LIM_INF_M2).sum())
n_alto = int((MZ_P.area_m2 > LIM_SUP_M2).sum())
MZ = MZ_P[(MZ_P.area_m2 >= LIM_INF_M2) & (MZ_P.area_m2 <= LIM_SUP_M2)].copy()

print(f"\nExcluidas por área < {LIM_INF_M2:,.0f} m²:  {n_bajo} "
      f"({100*n_bajo/len(MZ_P):.2f} %)")
print(f"Excluidas por área > {LIM_SUP_M2:,.0f} m²: {n_alto} "
      f"({100*n_alto/len(MZ_P):.2f} %)")
print(f"Manzanas analizadas: {len(MZ)} de {len(MZ_P)}")

Registros: 44518
CRS del archivo: EPSG:4326
Tipos de geometría: {'Polygon': 44448, 'MultiPolygon': 70}
Geometrías inválidas reparadas: 8
Descartadas por no ser polígono tras reparar: 0
Inválidas restantes: 0

Excluidas por área < 100 m²:  176 (0.40 %)
Excluidas por área > 100,000 m²: 244 (0.55 %)
Manzanas analizadas: 44098 de 44518


# Dimensiones por rectángulo mínimo rotado

In [6]:
def lados_rectangulo_minimo(geom):
    """Lado corto y largo del rectángulo mínimo rotado.

    Mide 'ancho x largo' con independencia de la orientación de la trama, a
    diferencia de la caja alineada a los ejes. Devuelve (nan, nan) si el
    polígono degenera en línea o punto, en vez de reventar.
    """
    rect = geom.minimum_rotated_rectangle
    if rect.geom_type != "Polygon" or rect.is_empty:
        return np.nan, np.nan
    x, y = rect.exterior.coords.xy
    lados = sorted(np.hypot(x[i + 1] - x[i], y[i + 1] - y[i]) for i in range(4))
    return lados[0], lados[2]


print("Calculando rectángulos mínimos… (~1 min)")
dims = np.array([lados_rectangulo_minimo(g) for g in MZ.geometry])
MZ["lado_corto_m"] = dims[:, 0]
MZ["lado_largo_m"] = dims[:, 1]
MZ["diagonal_m"]   = np.hypot(MZ.lado_corto_m, MZ.lado_largo_m)

n_degenerada = int(MZ.lado_largo_m.isna().sum())
if n_degenerada:
    print(f"Geometrías degeneradas sin rectángulo: {n_degenerada} (excluidas)")
    MZ = MZ[MZ.lado_largo_m.notna()].copy()

PCTS = [.05, .25, .5, .75, .90, .95]
print("\n--- Dimensiones de manzana, Bogotá completa (m) ---")
print(MZ[["lado_corto_m", "lado_largo_m", "diagonal_m",
          "area_m2"]].describe(percentiles=PCTS).round(1).to_string())

Calculando rectángulos mínimos… (~1 min)

--- Dimensiones de manzana, Bogotá completa (m) ---
       lado_corto_m  lado_largo_m  diagonal_m  area_m2
count       44098.0       44098.0     44098.0  44098.0
mean           46.5         100.7       112.7   4980.2
std            38.4          72.1        79.3   8562.5
min             1.0          10.7        14.7    100.7
5%             16.0          32.8        39.4    533.1
25%            24.0          59.8        66.7   1413.9
50%            34.9          84.9        94.6   2502.5
75%            54.2         116.5       128.7   4926.7
90%            87.0         169.7       189.6   9836.7
95%           118.8         225.3       256.5  17319.2
max           823.9        1208.8      1209.0  99781.4


# Separación entre manzanas y banda de tolerancia

In [7]:
# ---------------------------------------------------------------------------
# ¿Cuánto separa a dos manzanas vecinas en la capa catastral?
#
# ADVERTENCIA DE INTERPRETACIÓN: esta medida NO es el ancho de vía. Cerca de
# un tercio de las vecinas son contiguas o casi, porque la capa separa
# manzanas por línea de predio y no por eje vial. La mediana resultante queda
# muy por debajo del ancho real de una calle bogotana.
#
# Sirve para una sola cosa: construir una banda de tolerancias de asignación
# que no dependa de los umbrales de B1, que son justamente lo que B2 evalúa.
# Los veredictos se calculan sobre toda la banda y solo se adopta lo que no
# cambia dentro de ella.
# ---------------------------------------------------------------------------

MUESTRA_VIA = 3000
SEMILLA = 20260829          # fija, para que la medición sea reproducible
CORTE_CONTIGUIDAD_M = 2.0   # por debajo de esto, los polígonos se consideran
                            # contiguos: no hay calle entre ellos

base = MZ.reset_index(drop=True)
geoms = list(base.geometry.values)
arbol = STRtree(geoms)

rng = np.random.default_rng(SEMILLA)
idx = rng.choice(len(geoms), size=min(MUESTRA_VIA, len(geoms)), replace=False)

separaciones = []
for i in idx:
    vecino = arbol.query_nearest(geoms[i], exclusive=True, all_matches=False)
    j = int(np.atleast_1d(vecino)[0])
    separaciones.append(geoms[i].distance(geoms[j]))

SEP = pd.Series(separaciones, name="separacion_m")
SEP_NO_CONTIGUA = SEP[SEP >= CORTE_CONTIGUIDAD_M]

SEPARACION_MEDIANA_MZ = float(SEP.median())
SEPARACION_NO_CONTIGUA_MED = float(SEP_NO_CONTIGUA.median())
SEPARACION_P90 = float(SEP.quantile(0.90))

print(f"Manzanas muestreadas: {len(SEP)} (semilla {SEMILLA})")
print(SEP.describe(percentiles=PCTS).round(1).to_string())
print(f"\nContiguas o casi (<{CORTE_CONTIGUIDAD_M:.0f} m): "
      f"{int((SEP < CORTE_CONTIGUIDAD_M).sum())} de {len(SEP)} "
      f"({100*(SEP < CORTE_CONTIGUIDAD_M).mean():.1f} %)")
print(f"Separación mediana, todas las vecinas:      {SEPARACION_MEDIANA_MZ:5.1f} m")
print(f"Separación mediana, solo las no contiguas:  {SEPARACION_NO_CONTIGUA_MED:5.1f} m")
print(f"Percentil 90 de la separación:              {SEPARACION_P90:5.1f} m")
print("\nNinguno de estos valores es el ancho de una calle bogotana: la capa "
      "catastral separa manzanas por línea de predio, no por eje vial. Se usan "
      "como referencia de escala, no como medida de vía.")

# La tolerancia de asignación no se fija con un solo número. Se declara una
# banda y la celda de veredictos reporta qué conclusiones sobreviven a
# moverla. El valor adoptado es el extremo superior de la banda medida.
BANDA_TOLERANCIA_M = [
    round(SEPARACION_MEDIANA_MZ / 2, 1),
    round(SEPARACION_NO_CONTIGUA_MED, 1),
    round(SEPARACION_P90, 1),
    round(SEPARACION_P90 * 2, 1),
]
TOL_CONTENCION_M = max(BANDA_TOLERANCIA_M)
print(f"\nBanda de tolerancia evaluada: {BANDA_TOLERANCIA_M} m")
print(f"Tolerancia adoptada: {TOL_CONTENCION_M:.1f} m "
      "(extremo superior de la banda; ninguno de sus valores proviene de los "
      "umbrales de B1)")

Manzanas muestreadas: 3000 (semilla 20260829)
count    3000.0
mean        5.1
std         4.1
min         0.0
5%          0.0
25%         0.9
50%         5.4
75%         8.0
90%        10.8
95%        11.8
max        26.1

Contiguas o casi (<2 m): 919 de 3000 (30.6 %)
Separación mediana, todas las vecinas:        5.4 m
Separación mediana, solo las no contiguas:    6.4 m
Percentil 90 de la separación:               10.8 m

Ninguno de estos valores es el ancho de una calle bogotana: la capa catastral separa manzanas por línea de predio, no por eje vial. Se usan como referencia de escala, no como medida de vía.

Banda de tolerancia evaluada: [2.7, 6.4, 10.8, 21.5] m
Tolerancia adoptada: 21.5 m (extremo superior de la banda; ninguno de sus valores proviene de los umbrales de B1)


# Escala en el entorno de las estaciones

In [8]:
# El umbral debe reflejar la trama DONDE hay estaciones, no el promedio de la
# ciudad entera, que incluye periferia con manzanas muy distintas.
EST = pd.concat([
    OCM_P[OCM_P.ciudad == "Bogota"][["geometry"]],
    OSM_P[OSM_P.ciudad == "Bogota"][["geometry"]],
], ignore_index=True)
EST = gpd.GeoDataFrame(EST, geometry="geometry", crs=CRS_PROY)

zona = EST.buffer(RADIO_LOCAL_M).union_all()
MZ_LOCAL = MZ[MZ.intersects(zona)].copy()

print(f"Estaciones de Bogotá consideradas: {len(EST)}")
print(f"Manzanas a menos de {RADIO_LOCAL_M:.0f} m de alguna: {len(MZ_LOCAL)}")
print("\n--- Dimensiones en el entorno de las estaciones (m) ---")
print(MZ_LOCAL[["lado_corto_m", "lado_largo_m",
                "diagonal_m"]].describe(percentiles=PCTS).round(1).to_string())

# Sensibilidad del propio parámetro de B2: si la mediana se mueve poco al
# cambiar el radio, el anclaje no depende de haber elegido 1.000 m.
print("\n--- Estabilidad del anclaje frente al radio del entorno ---")
filas_radio = []
for radio in (250, 500, 1000, 2000, 5000):
    sub = MZ[MZ.intersects(EST.buffer(radio).union_all())]
    filas_radio.append({
        "radio_m": radio,
        "manzanas": len(sub),
        "lado_corto_med": round(sub.lado_corto_m.median(), 1),
        "lado_largo_med": round(sub.lado_largo_m.median(), 1),
    })
SENS_RADIO = pd.DataFrame(filas_radio)
print(SENS_RADIO.to_string(index=False))
print(f"\nCiudad completa — lado largo mediano: {MZ.lado_largo_m.median():.1f} m")

Estaciones de Bogotá consideradas: 69
Manzanas a menos de 1000 m de alguna: 8305

--- Dimensiones en el entorno de las estaciones (m) ---
       lado_corto_m  lado_largo_m  diagonal_m
count        8305.0        8305.0      8305.0
mean           61.0         120.1       136.8
std            40.7          76.8        83.6
min             3.3          12.3        16.7
5%             20.3          42.9        51.8
25%            37.9          77.9        89.8
50%            50.3         101.7       116.8
75%            74.1         139.7       156.6
90%           104.6         200.8       225.5
95%           137.1         257.6       291.4
max           823.9         952.9      1166.7

--- Estabilidad del anclaje frente al radio del entorno ---
 radio_m  manzanas  lado_corto_med  lado_largo_med
     250      1058            61.1           109.9
     500      3150            56.5           107.3
    1000      8305            50.3           101.7
    2000     16915            45.1           

# Justificación de umbrales

In [9]:
med_corto = MZ_LOCAL.lado_corto_m.median()
med_largo = MZ_LOCAL.lado_largo_m.median()
med_diag  = MZ_LOCAL.diagonal_m.median()

print("=" * 76)
print("JUSTIFICACIÓN DE UMBRALES")
print("=" * 76)
print(f"""
Escala medida sobre la capa catastral (entorno de estaciones, mediana de
{len(MZ_LOCAL)} manzanas):
  lado corto  {med_corto:6.1f} m
  lado largo  {med_largo:6.1f} m
  diagonal    {med_diag:6.1f} m

Separación entre manzanas vecinas: mediana {SEPARACION_MEDIANA_MZ:.1f} m,
percentil 90 {SEPARACION_P90:.1f} m. Esta magnitud NO es el ancho de una vía:
la capa catastral separa manzanas por línea de predio y una fracción
importante de las vecinas son contiguas. Se usa como referencia de escala de
la trama, no como medida de calle, y no interviene en la justificación de los
umbrales de clasificación.

UMBRAL C1/C2 = {UMBRAL_CONFIRMADA:.0f} m — ANCLAJE EXTERNO
  Error horizontal de receptores GPS de consumo en entorno urbano: 7 a 13 m
  (Merry y Bettinger, 2019, PLOS ONE, doi:10.1371/journal.pone.0219890). El
  umbral equivale al doble del extremo superior de ese rango, de modo que dos
  registros de una misma estación caben dentro aun con el peor error en
  ambas fuentes.
  Contraste con la trama: {UMBRAL_CONFIRMADA/med_corto:.2f} del lado corto mediano de manzana,
  es decir, muy dentro de un mismo predio.

UMBRAL C2/C3 = {UMBRAL_PROBABLE:.0f} m — ANCLAJE INTERNO, MEDIDO
  Corresponde al lado largo mediano de la manzana en el entorno de las
  estaciones ({med_largo:.1f} m). Razón umbral / lado largo: {UMBRAL_PROBABLE/med_largo:.2f}.
  Lectura: dos registros separados por más de una manzana no describen el
  mismo sitio.

RADIO_DUPLICADO = {RADIO_DUPLICADO:.0f} m — ANCLAJE INTERNO
  Fusión de duplicados dentro de una misma fuente. Referencia: {RADIO_DUPLICADO/med_corto:.2f} del
  lado corto mediano de manzana ({med_corto:.1f} m). Dos puntos de una misma
  fuente a menos de esa distancia están dentro del mismo predio o en predios
  colindantes.

RADIO_SECUNDARIO = {RADIO_SECUNDARIO:.0f} m — NO VINCULANTE (se verifica abajo)
RADIO_BUSQUEDA_MAX = {RADIO_BUSQUEDA_MAX:.0f} m — NO VINCULANTE (se verifica abajo)
  Son radios de corte del candidato, no criterios de clasificación. La
  justificación adecuada no es geométrica sino empírica: mostrar que el
  resultado no cambia dentro de un rango amplio. Eso se mide en la celda de
  sensibilidad.

TOLERANCIA DE CONTENCIÓN — BANDA, NO VALOR ÚNICO
  La asignación de un punto a su manzana no cuelga de un número elegido. Se
  evalúa sobre la banda {BANDA_TOLERANCIA_M} m y solo se adoptan
  los veredictos que no cambian dentro de ella. Ningún valor de la banda
  proviene de los umbrales de B1, de modo que la verificación es
  independiente de lo que se está verificando.

LIMITACIÓN QUE DEBE DECLARARSE
  El anclaje se mide sobre la capa catastral de Bogotá (UAECD). No existe
  equivalente abierto de manzana catastral para Medellín en las mismas
  condiciones, de modo que el umbral se transfiere sin verificación local.
  Medellín aporta solo 2 de los 20 pares primarios, así que el efecto es
  acotado, pero el límite se declara en el informe.
""")

JUSTIFICACIÓN DE UMBRALES

Escala medida sobre la capa catastral (entorno de estaciones, mediana de
8305 manzanas):
  lado corto    50.3 m
  lado largo   101.7 m
  diagonal     116.8 m

Separación entre manzanas vecinas: mediana 5.4 m,
percentil 90 10.8 m. Esta magnitud NO es el ancho de una vía:
la capa catastral separa manzanas por línea de predio y una fracción
importante de las vecinas son contiguas. Se usa como referencia de escala de
la trama, no como medida de calle, y no interviene en la justificación de los
umbrales de clasificación.

UMBRAL C1/C2 = 25 m — ANCLAJE EXTERNO
  Error horizontal de receptores GPS de consumo en entorno urbano: 7 a 13 m
  (Merry y Bettinger, 2019, PLOS ONE, doi:10.1371/journal.pone.0219890). El
  umbral equivale al doble del extremo superior de ese rango, de modo que dos
  registros de una misma estación caben dentro aun con el peor error en
  ambas fuentes.
  Contraste con la trama: 0.50 del lado corto mediano de manzana,
  es decir, muy dentro de u

# Análisis de sensibilidad

In [10]:
# ---------------------------------------------------------------------------
# 1. Reimplementación independiente de la regla de B1.
def evidencia_textual(sim_nombre, sim_operador, umbral):
    a = pd.notna(sim_nombre) and sim_nombre >= umbral
    b = pd.notna(sim_operador) and sim_operador >= umbral
    return bool(a or b)


def clasificar(distancia, evid, u_conf, u_prob):
    if distancia <= u_conf:
        return "C1"
    if distancia <= u_prob:
        return "C2"
    return "C3" if evid else "DEGRADADO"


def clasificar_todo(u_conf, u_prob, u_txt):
    return pd.Series([
        clasificar(f.distancia_m,
                   evidencia_textual(f.similitud_nombre, f.similitud_operador, u_txt),
                   u_conf, u_prob)
        for f in PARES.itertuples()
    ], index=PARES.index)


ADOPTADA = clasificar_todo(UMBRAL_CONFIRMADA, UMBRAL_PROBABLE, UMBRAL_TEXTUAL)
B1_CORTA = PARES.categoria.str.split().str[0].str.replace(
    "DEGRADADO", "DEGRADADO", regex=False)

discrepancias = int((ADOPTADA != B1_CORTA).sum())
print(f"Reimplementación de la regla de B1: {len(PARES) - discrepancias} "
      f"de {len(PARES)} coinciden")
if discrepancias:
    print(PARES.assign(b1=B1_CORTA, b2=ADOPTADA)
              .loc[ADOPTADA != B1_CORTA,
                   ["ocm_id", "osm_uid", "distancia_m", "b1", "b2"]]
              .to_string(index=False))

# ---------------------------------------------------------------------------
# 2. Saltos de la distribución: dónde hay huecos naturales.
# ---------------------------------------------------------------------------
d = np.sort(PARES.distancia_m.values)
print("\n--- Saltos en la distribución de distancias emparejadas ---")
for i in range(1, len(d)):
    marca = "   <== hueco" if d[i] - d[i-1] > 30 else ""
    print(f"  {d[i-1]:7.1f} -> {d[i]:7.1f}   salto {d[i]-d[i-1]:6.1f}{marca}")

# ---------------------------------------------------------------------------
# 3. Cuántas estaciones cambian de categoría al mover cada umbral.
#    Esto es lo que pidió la tutoría: no conteos por umbral, sino el cambio.
# ---------------------------------------------------------------------------
filas = []
for u_conf in (10, 15, 20, 25, 30, 40, 50):
    cl = clasificar_todo(u_conf, UMBRAL_PROBABLE, UMBRAL_TEXTUAL)
    filas.append({"umbral": "C1/C2", "valor": u_conf,
                  "cambian": int((cl != ADOPTADA).sum()),
                  **cl.value_counts().to_dict()})
for u_prob in (60, 80, 100, 120, 150, 200):
    cl = clasificar_todo(UMBRAL_CONFIRMADA, u_prob, UMBRAL_TEXTUAL)
    filas.append({"umbral": "C2/C3", "valor": u_prob,
                  "cambian": int((cl != ADOPTADA).sum()),
                  **cl.value_counts().to_dict()})
for u_txt in (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60):
    cl = clasificar_todo(UMBRAL_CONFIRMADA, UMBRAL_PROBABLE, u_txt)
    filas.append({"umbral": "textual", "valor": u_txt,
                  "cambian": int((cl != ADOPTADA).sum()),
                  **cl.value_counts().to_dict()})

SENS = pd.DataFrame(filas).fillna(0)
for c in ("C1", "C2", "C3", "DEGRADADO"):
    if c in SENS:
        SENS[c] = SENS[c].astype(int)
print("\n--- Estaciones que cambian de categoría al mover cada umbral ---")
print(SENS.to_string(index=False))

# ---------------------------------------------------------------------------
# 4. Los radios pendientes: 500 m y 300 m. La prueba de que un radio está
#    bien puesto es que el resultado no dependa de él.
# ---------------------------------------------------------------------------

def pares_mutuos(radio):
    """Emparejamiento recíproco por vecino más cercano, por ciudad.

    Reimplementación mínima (sin deduplicación interna) usada solo para
    sensibilidad. La diferencia esperada con B1 son los casos que B1 fusiona
    antes de emparejar.
    """
    salida = []
    for ciudad in ("Bogota", "Medellin"):
        a = OCM_P[OCM_P.ciudad == ciudad]
        b = OSM_P[OSM_P.ciudad == ciudad]
        if a.empty or b.empty:
            continue
        ax = np.c_[a.geometry.x, a.geometry.y]
        bx = np.c_[b.geometry.x, b.geometry.y]
        D = np.hypot(ax[:, 0][:, None] - bx[:, 0][None, :],
                     ax[:, 1][:, None] - bx[:, 1][None, :])
        j_de_a, i_de_b = D.argmin(1), D.argmin(0)
        for i, j in enumerate(j_de_a):
            if i_de_b[j] == i and D[i, j] <= radio:
                salida.append((a.index[i], b.index[j], D[i, j]))
    return pd.DataFrame(salida, columns=["ocm_id", "osm_uid", "distancia_m"])

print("\n--- Sensibilidad del radio de búsqueda ---")
referencia = None
for radio in (100, 200, 300, 500, 1000, 2000, 5000):
    pm = pares_mutuos(radio)
    clave = set(zip(pm.ocm_id, pm.osm_uid))
    if radio == RADIO_BUSQUEDA_MAX:
        referencia = clave
    print(f"  {radio:5d} m -> {len(pm):2d} pares mutuos")
extra = {r: len(set(zip(*pares_mutuos(r)[['ocm_id', 'osm_uid']].values.T)) - referencia)
         for r in (1000, 2000, 5000)}
print(f"Pares nuevos al ampliar más allá de {RADIO_BUSQUEDA_MAX:.0f} m: {extra}")
print("Si esos pares nuevos quedan todos por encima del umbral C2/C3, el "
      "radio no es vinculante: recorta candidatos que igual serían DEGRADADO.")

# --- Radio de duplicado interno: dónde está el hueco natural ---
print("\n--- Sensibilidad del radio de duplicado interno ---")
for nombre, G in (("OCM", OCM_P), ("OSM", OSM_P)):
    for ciudad in ("Bogota", "Medellin"):
        g = G[G.ciudad == ciudad]
        if len(g) < 2:
            continue
        xy = np.c_[g.geometry.x, g.geometry.y]
        D = np.hypot(xy[:, 0][:, None] - xy[:, 0][None, :],
                     xy[:, 1][:, None] - xy[:, 1][None, :])
        np.fill_diagonal(D, np.inf)
        cercanos = np.sort(D[np.triu_indices(len(g), 1)])
        cercanos = cercanos[cercanos < 500]
        conteos = {r: int((cercanos <= r).sum()) for r in (10, 25, 50, 75, 100, 200)}
        print(f"  {nombre} {ciudad:9s} pares internos por radio: {conteos}")

Reimplementación de la regla de B1: 20 de 20 coinciden

--- Saltos en la distribución de distancias emparejadas ---
      1.7 ->    10.2   salto    8.4
     10.2 ->    15.8   salto    5.6
     15.8 ->    15.9   salto    0.1
     15.9 ->    23.6   salto    7.7
     23.6 ->    27.5   salto    4.0
     27.5 ->    28.2   salto    0.6
     28.2 ->    30.5   salto    2.3
     30.5 ->    32.0   salto    1.5
     32.0 ->    41.2   salto    9.2
     41.2 ->    43.0   salto    1.9
     43.0 ->    46.9   salto    3.8
     46.9 ->    79.5   salto   32.6   <== hueco
     79.5 ->    82.5   salto    3.0
     82.5 ->   139.0   salto   56.5   <== hueco
    139.0 ->   167.3   salto   28.3
    167.3 ->   183.1   salto   15.8
    183.1 ->   295.1   salto  112.1   <== hueco
    295.1 ->   389.0   salto   93.8   <== hueco
    389.0 ->   456.7   salto   67.7   <== hueco

--- Estaciones que cambian de categoría al mover cada umbral ---
 umbral  valor  cambian  C2  DEGRADADO  C3  C1
  C1/C2  10.00        4  13

# Verificación de la reproyección

In [11]:
lat_ocm = OCM.loc[PARES.ocm_id, "lat"].to_numpy(float)
lon_ocm = OCM.loc[PARES.ocm_id, "lon"].to_numpy(float)
lat_osm = OSM.loc[PARES.osm_uid, "lat"].to_numpy(float)
lon_osm = OSM.loc[PARES.osm_uid, "lon"].to_numpy(float)

# Camino independiente 1: geodésica sobre GRS80, el elipsoide de
# MAGNA-SIRGAS. Es la referencia correcta contra EPSG:9377.
_, _, d_geodesica = GEOD.inv(lon_ocm, lat_ocm, lon_osm, lat_osm)

# Camino independiente 2: cambio de datum 4326 -> 4686 con las mismas
# coordenadas, para cuantificar la discrepancia de la ficha de la capa.
tr4686 = Transformer.from_crs("EPSG:4686", CRS_PROY, always_xy=True)
x1, y1 = tr4686.transform(lon_ocm, lat_ocm)
x2, y2 = tr4686.transform(lon_osm, lat_osm)

# Camino 3: distancia ingenua en grados, SIN coseno de latitud. No es un
# método alternativo: es la reproducción del error del Bloque A, que se deja
# medido para poder citarlo en el informe.
d_grados = np.hypot(lat_osm - lat_ocm, lon_osm - lon_ocm) * 111_320

VD = PARES[["ciudad", "ocm_id", "osm_uid"]].copy()
VD["d_b1_9377"]    = PARES.distancia_m.values
VD["d_geodesica"]  = d_geodesica
VD["d_4686_9377"]  = np.hypot(x2 - x1, y2 - y1)
VD["d_grados_sin_coseno"] = d_grados
VD["sesgo_ppm"] = 1e6 * (VD.d_b1_9377 - VD.d_geodesica) / VD.d_geodesica

print("--- Distancias por método (m) ---")
print(VD.drop(columns=["ciudad", "osm_uid"]).round(2).to_string(index=False))

print("\n--- Sesgo de EPSG:9377 contra la geodésica, por ciudad (ppm) ---")
print(VD.groupby("ciudad").sesgo_ppm.describe()[["count", "mean", "min", "max"]]
        .round(1).to_string())
print(f"\nEfecto sobre el umbral de {UMBRAL_CONFIRMADA:.0f} m: "
      f"{UMBRAL_CONFIRMADA * VD.sesgo_ppm.abs().max() / 1e6 * 100:.2f} cm. "
      "Es deformación de la proyección, no error de cálculo: dentro de cada "
      "ciudad el factor es casi constante, y su magnitud depende de la "
      "distancia al meridiano central.")

print("\n--- Discrepancia de datum 4326 vs 4686 ---")
dif = (VD.d_4686_9377 - VD.d_b1_9377).abs()
print(f"  diferencia máxima: {dif.max()*1000:.3f} mm — irrelevante frente a "
      f"un umbral de {UMBRAL_CONFIRMADA:.0f} m")

print("\n--- Error del método en grados (el defecto corregido en el Bloque A) ---")
err = 100 * (VD.d_grados_sin_coseno - VD.d_geodesica) / VD.d_geodesica
print(f"  error medio {err.abs().mean():.1f} %  máximo {err.abs().max():.1f} %")

# --- Unidades de SHAPE_AREA: se verifica, no se afirma ---
if {"SHAPE_AREA", "MANCODIGO"} <= set(MZ_P.columns):
    cmp = MZ_P.loc[MZ_P.SHAPE_AREA > 0, ["MANCODIGO", "SHAPE_AREA", "area_m2"]].head(500).copy()
    cmp["razon"] = cmp.area_m2 / cmp.SHAPE_AREA
    cv = cmp.razon.std() / cmp.razon.mean()
    print(f"\n--- Campo SHAPE_AREA ---")
    print(f"  razón m² / SHAPE_AREA: mediana {cmp.razon.median():.3e}, "
          f"coef. de variación {cv:.4f}")
    print("  Razón casi constante y del orden de 1e10: SHAPE_AREA viene en "
          "grados cuadrados, no es un área métrica.")
    RAZON_SHAPE_AREA_CV = float(cv)
else:
    RAZON_SHAPE_AREA_CV = None
    print("\nLa capa no trae SHAPE_AREA; se omite la verificación de unidades.")

--- Distancias por método (m) ---
ocm_id  d_b1_9377  d_geodesica  d_4686_9377  d_grados_sin_coseno  sesgo_ppm
475409     456.70       456.99       456.70               459.95    -627.09
475140     183.07       183.18       183.07               183.86    -630.80
475122      79.50        79.54        79.50                79.86    -618.77
475120      30.48        30.50        30.48                30.70    -596.96
472671      27.54        27.56        27.54                27.73    -637.59
470940      82.51        82.57        82.51                82.85    -615.37
470939     295.13       295.31       295.13               296.39    -610.98
313651     388.97       389.21       388.97               391.56    -619.95
311258      43.03        43.06        43.03                43.34    -615.92
302435      10.16        10.17        10.16                10.22    -640.33
302196     167.28       167.39       167.28               168.28    -633.95
302195      28.19        28.20        28.19           

In [12]:
# --- El sesgo de la proyección se predice, no solo se observa ---
# EPSG:9377 es Transversa de Mercator con meridiano central en -73° y
# k0 = 0,9992. El factor de escala crece con la distancia al meridiano: vale
# 1 a unos 255 km y más allá pasa de 1. Por eso el sesgo es NEGATIVO en
# Bogotá (~119 km) y POSITIVO en Medellín (~284 km): es geometría, no ruido.
#
# La fórmula es la aproximación de primer orden. El término omitido crece con
# la cuarta potencia de la distancia al meridiano, de ahí que el residuo de
# Medellín (7,5 ppm) sea mayor que el de Bogotá (1,2 ppm).

K0_9377, LON_CM_9377, R_EQ = 0.9992, -73.0, 6378137.0


def ppm_predicho(lon, lat):
    x = np.radians(lon - LON_CM_9377) * R_EQ * np.cos(np.radians(lat))
    return (K0_9377 * (1 + x ** 2 / (2 * R_EQ ** 2)) - 1) * 1e6


VD["ppm_predicho"] = ppm_predicho((lon_ocm + lon_osm) / 2,
                                  (lat_ocm + lat_osm) / 2)
VD["ppm_residuo"] = VD.sesgo_ppm - VD.ppm_predicho

print("\n--- Sesgo observado vs predicho por la proyección (ppm) ---")
print(VD.groupby("ciudad")[["sesgo_ppm", "ppm_predicho", "ppm_residuo"]]
        .agg(["count", "mean", "min", "max"]).round(1).to_string())


--- Sesgo observado vs predicho por la proyección (ppm) ---
         sesgo_ppm                      ppm_predicho                      ppm_residuo               
             count   mean    min    max        count   mean    min    max       count mean  min  max
ciudad                                                                                              
Bogota          18 -623.6 -640.3 -597.0           18 -624.8 -641.4 -598.3          18  1.2  1.1  1.4
Medellin         2  204.7  200.0  209.4            2  197.2  192.6  201.9           2  7.5  7.4  7.5


# Contención en manzana catastral

In [13]:
# Verificación con una TERCERA fuente: la manzana catastral no proviene de
# OpenStreetMap ni de OpenChargeMap, de modo que no hay circularidad. La
# tolerancia tampoco: se midió sobre esta misma capa (media vía).
filas = []
for caso, (oid, ouid) in CASOS.items():
    a, b = OCM.loc[oid], OSM.loc[ouid]
    filas += [{"caso": caso, "punto": "OCM", "id": oid, "lat": a.lat, "lon": a.lon},
              {"caso": caso, "punto": "OSM", "id": ouid, "lat": b.lat, "lon": b.lon}]

PTS = gpd.GeoDataFrame(
    pd.DataFrame(filas),
    geometry=gpd.points_from_xy([f["lon"] for f in filas],
                                [f["lat"] for f in filas]),
    crs=CRS_GEO,
).to_crs(CRS_PROY)

# Se usa MZ_P (sin filtrar por área) para no perder manzanas grandes legítimas.
MZ_GEO = MZ_P[["MANCODIGO", "SECCODIGO", "geometry"]].copy()
MZ_GEO["area_mz_m2"] = MZ_P.area

asig = gpd.sjoin_nearest(PTS, MZ_GEO, how="left", distance_col="dist_mz_m")

# sjoin_nearest devuelve más de una fila cuando hay empate de distancia.
# Se resuelve de forma determinista por el código de manzana y se declara.
n_empates = int(len(asig) - len(PTS))
if n_empates:
    print(f"Empates de vecino más cercano: {n_empates}. "
          "Se conserva la manzana de menor MANCODIGO (criterio determinista).")
    asig = (asig.sort_values(["caso", "punto", "dist_mz_m", "MANCODIGO"])
                .groupby(["caso", "punto"], as_index=False).first())
if len(asig) != len(PTS):
    raise RuntimeError("La asignación no dejó exactamente una manzana por punto.")

asig["dentro"]   = asig.dist_mz_m <= 1e-6
asig["asignada"] = asig.dist_mz_m <= TOL_CONTENCION_M

print("\n--- Asignación punto a punto ---")
print(asig[["caso", "punto", "MANCODIGO", "area_mz_m2", "dist_mz_m",
            "dentro", "asignada"]].round(1).to_string(index=False))
print(f"\nPuntos fuera de toda manzana pero dentro de la tolerancia de "
      f"{TOL_CONTENCION_M:.1f} m: {int((~asig.dentro & asig.asignada).sum())}")


--- Asignación punto a punto ---
caso punto MANCODIGO  area_mz_m2  dist_mz_m  dentro  asignada
C3-1   OCM 005106010     20403.6        6.6   False      True
C3-1   OSM 005106010     20403.6        0.0    True      True
C3-2   OCM 008311007     20586.1        0.0    True      True
C3-2   OSM 008312017      7059.3        0.0    True      True
C3-3   OCM 006319011     16273.8        0.0    True      True
C3-3   OSM 006319001    154881.1        0.0    True      True
N1-3   OCM 004208010     17630.9       15.6   False      True
N1-3   OSM 004208011     20257.9        0.0    True      True
N1-2   OCM 008415001    123946.6        0.0    True      True
N1-2   OSM 008415001    123946.6        0.0    True      True
N1-1   OCM 005506001   1145790.8        0.0    True      True
N1-1   OSM 005506001   1145790.8        0.0    True      True

Puntos fuera de toda manzana pero dentro de la tolerancia de 21.5 m: 2


# Veredictos con criterio de suficiencia

In [14]:
# La contención solo es evidencia si el polígono discrimina. Un polígono por
# encima del corte de manzana urbana contiene casi cualquier par de puntos
# del sector y no permite concluir: se declara indeterminado.
#
# Además, el veredicto no puede depender de la tolerancia elegida. Se calcula
# sobre toda la banda y solo se adopta lo que resulta estable en ella.

def veredictos(tolerancia):
    """Veredictos de contención para una tolerancia de asignación dada."""
    salida = []
    for caso in CASOS:
        s = asig[asig.caso == caso]
        a = s[s.punto == "OCM"].iloc[0]
        b = s[s.punto == "OSM"].iloc[0]
        ok = (a.dist_mz_m <= tolerancia) and (b.dist_mz_m <= tolerancia)
        if not ok:
            ver = None
            mot = (f"punto a más de {tolerancia:.1f} m de toda manzana "
                   f"(OCM {a.dist_mz_m:.1f} m, OSM {b.dist_mz_m:.1f} m)")
        elif a.MANCODIGO != b.MANCODIGO:
            ver = False
            mot = (f"manzanas distintas {a.MANCODIGO} ({a.area_mz_m2:,.0f} m²) "
                   f"vs {b.MANCODIGO} ({b.area_mz_m2:,.0f} m²)")
        elif a.area_mz_m2 > LIM_SUP_M2:
            ver = None
            mot = (f"misma manzana {a.MANCODIGO} pero de {a.area_mz_m2:,.0f} m², "
                   f"por encima del corte de manzana urbana "
                   f"({LIM_SUP_M2:,.0f} m²): la contención no discrimina")
        else:
            ver = True
            mot = f"misma manzana {a.MANCODIGO} ({a.area_mz_m2:,.0f} m²)"
        salida.append({"caso": caso, "veredicto": ver, "motivo": mot,
                       "ocm_id": a.id, "osm_uid": b.id,
                       "area_mz_m2": round(a.area_mz_m2, 1),
                       "dist_ocm_m": round(a.dist_mz_m, 1),
                       "dist_osm_m": round(b.dist_mz_m, 1)})
    return pd.DataFrame(salida).sort_values("caso").reset_index(drop=True)


# --- Sensibilidad del veredicto a la tolerancia ---------------------------
print("--- Veredictos por tolerancia ---")
tabla = {}
for tol in BANDA_TOLERANCIA_M:
    tabla[f"{tol:.1f} m"] = veredictos(tol).set_index("caso").veredicto
SENS_TOL = pd.DataFrame(tabla)
print(SENS_TOL.to_string())

estables = SENS_TOL.apply(
    lambda f: bool(f.notna().all() and f.nunique(dropna=True) == 1), axis=1)
print(f"\nCasos con veredicto estable en toda la banda: "
      f"{int(estables.sum())} de {len(SENS_TOL)} -> "
      f"{sorted(SENS_TOL.index[estables])}")
print("Los casos que cambian de veredicto al mover la tolerancia NO se dan "
      "por resueltos: su conclusión dependería de un parámetro y no del dato.")

# --- Veredicto adoptado ---------------------------------------------------
RES = veredictos(TOL_CONTENCION_M)
RES["estable_en_banda"] = RES.caso.map(estables)
inestable = ~RES.estable_en_banda
RES.loc[inestable, "motivo"] = (
    RES.loc[inestable, "motivo"]
    + " — NO ADOPTADO: el veredicto cambia dentro de la banda de tolerancia")
RES.loc[inestable, "veredicto"] = None

print(f"\n--- Veredicto adoptado (tolerancia {TOL_CONTENCION_M:.1f} m) ---")
for f in RES.itertuples():
    print(f"  {f.caso}  {str(f.veredicto):5s}  {f.motivo}")
print(f"\nResueltos: {int(RES.veredicto.notna().sum())} de {len(RES)}")

--- Veredictos por tolerancia ---
      2.7 m  6.4 m 10.8 m 21.5 m
caso                            
C3-1   None   None   True   True
C3-2  False  False  False  False
C3-3  False  False  False  False
N1-1   None   None   None   None
N1-2   None   None   None   None
N1-3   None   None   None  False

Casos con veredicto estable en toda la banda: 2 de 6 -> ['C3-2', 'C3-3']
Los casos que cambian de veredicto al mover la tolerancia NO se dan por resueltos: su conclusión dependería de un parámetro y no del dato.

--- Veredicto adoptado (tolerancia 21.5 m) ---
  C3-1  None   misma manzana 005106010 (20,404 m²) — NO ADOPTADO: el veredicto cambia dentro de la banda de tolerancia
  C3-2  False  manzanas distintas 008311007 (20,586 m²) vs 008312017 (7,059 m²)
  C3-3  False  manzanas distintas 006319011 (16,274 m²) vs 006319001 (154,881 m²)
  N1-1  None   misma manzana 005506001 pero de 1,145,791 m², por encima del corte de manzana urbana (100,000 m²): la contención no discrimina — NO ADOPTADO: el 

# Pruebas de aceptación

In [15]:
# Pruebas que pueden fallar. No se verifica que C1 cumpla la regla que define
# C1: eso es tautología. Cada prueba compara dos caminos distintos, o
# contrasta contra una fuente o una teoría externa al cálculo que se evalúa.

PRUEBAS = []


def prueba(nombre, condicion, detalle=""):
    PRUEBAS.append({"prueba": nombre,
                    "resultado": "PASA" if condicion else "FALLA",
                    "detalle": detalle})
    print(f"  {'PASA' if condicion else 'FALLA'}  {nombre}"
          f"{('  — ' + detalle) if detalle else ''}")


print("--- Pruebas de aceptación de B2 ---")

# --- 1. Distancias: camino independiente ---------------------------------
err_rel = 100 * (VD.d_b1_9377 - VD.d_geodesica).abs() / VD.d_geodesica
prueba("Las distancias de B1 se reproducen por geodésica GRS80 (<0,2 %)",
       bool((err_rel < 0.2).all()),
       f"desviación máxima {err_rel.max():.4f} %")

# --- 2. Proyección: el sesgo es deformación, no error de cálculo ----------
# El signo NO es el mismo en las dos ciudades: Bogotá subestima y Medellín
# sobreestima, según el lado de la línea de escala unitaria en que caen. Lo
# que debe ser constante es el signo dentro de cada ciudad.
prueba("El sesgo de EPSG:9377 tiene signo constante DENTRO de cada ciudad",
       bool(VD.groupby("ciudad").sesgo_ppm
              .apply(lambda s: (s > 0).all() or (s < 0).all()).all()),
       "; ".join(f"{c}: {g.sesgo_ppm.min():.0f} a {g.sesgo_ppm.max():.0f} ppm"
                 for c, g in VD.groupby("ciudad")))

# Tolerancia de 50 ppm: el residuo observado es de 8 ppm, y el margen deja
# espacio para el término de segundo orden que la fórmula no incluye.
prueba("El sesgo coincide con el factor de escala teórico de la proyección",
       bool((VD.ppm_residuo.abs() < 50).all()),
       f"residuo máximo {VD.ppm_residuo.abs().max():.1f} ppm")

prueba("La discrepancia de datum 4326/4686 es despreciable frente al umbral",
       bool(dif.max() < 0.01 * UMBRAL_CONFIRMADA),
       f"máximo {dif.max()*1000:.3f} mm")

# --- 3. Integridad de insumos y de la asignación --------------------------
prueba("El crudo de manzana coincide con su hash registrado",
       sha_actual == procedencia["sha256"])

prueba("Cada punto de los casos recibe exactamente una manzana",
       len(asig) == len(PTS),
       f"{len(asig)} filas para {len(PTS)} puntos")

# NUEVA: la separación catastral no se confunde con un ancho de vía.
prueba("La separación entre manzanas no se presenta como ancho de vía",
       SEPARACION_MEDIANA_MZ < 10.0,
       f"mediana {SEPARACION_MEDIANA_MZ:.1f} m, declarada como separación "
       "catastral y no como calle")

# NUEVA: ningún veredicto adoptado depende de la tolerancia elegida.
prueba("Todo veredicto adoptado es estable en la banda de tolerancia",
       bool(RES.loc[RES.veredicto.notna(), "estable_en_banda"].all()),
       f"{int(RES.veredicto.notna().sum())} adoptados de {len(RES)} "
       f"sobre la banda {BANDA_TOLERANCIA_M} m")

# --- 4. Lectura de la regla de B1 ----------------------------------------
prueba("La regla de clasificación se reimplementó sin discrepancias",
       discrepancias == 0, f"{discrepancias} pares difieren de B1")

# --- 5. Anclaje del umbral ------------------------------------------------
# La mediana NO es estable frente al radio: decrece de forma monótona porque
# la trama se hace más menuda hacia la periferia. Eso es un gradiente real,
# no ruido del método, y se verifica como tal.
prueba("La mediana varía de forma monótona con el radio (gradiente, no ruido)",
       bool(SENS_RADIO.sort_values("radio_m")
                      .lado_largo_med.is_monotonic_decreasing),
       f"medianas {SENS_RADIO.lado_largo_med.tolist()}")

prueba("El umbral C2/C3 cae dentro del rango de medianas de todos los radios",
       bool(SENS_RADIO.lado_largo_med.min() <= UMBRAL_PROBABLE
            <= SENS_RADIO.lado_largo_med.max()),
       f"rango {SENS_RADIO.lado_largo_med.min():.1f} a "
       f"{SENS_RADIO.lado_largo_med.max():.1f} m, umbral {UMBRAL_PROBABLE:.0f} m")

prueba("El umbral C2/C3 está dentro del ±15 % del lado largo mediano",
       abs(UMBRAL_PROBABLE / med_largo - 1) <= 0.15,
       f"razón {UMBRAL_PROBABLE/med_largo:.3f}")

# --- 6. Unidades de la capa catastral ------------------------------------
if RAZON_SHAPE_AREA_CV is not None:
    prueba("SHAPE_AREA no es métrico (razón casi constante)",
           RAZON_SHAPE_AREA_CV < 0.05, f"cv {RAZON_SHAPE_AREA_CV:.4f}")

# --------------------------------------------------------------------------
TESTS = pd.DataFrame(PRUEBAS)
n_falla = int((TESTS.resultado == "FALLA").sum())
print(f"\n{len(TESTS) - n_falla} de {len(TESTS)} pruebas pasan")
if n_falla:
    raise AssertionError(f"{n_falla} pruebas de aceptación fallaron; no se exporta.")

--- Pruebas de aceptación de B2 ---
  PASA  Las distancias de B1 se reproducen por geodésica GRS80 (<0,2 %)  — desviación máxima 0.0640 %
  PASA  El sesgo de EPSG:9377 tiene signo constante DENTRO de cada ciudad  — Bogota: -640 a -597 ppm; Medellin: 200 a 209 ppm
  PASA  El sesgo coincide con el factor de escala teórico de la proyección  — residuo máximo 7.5 ppm
  PASA  La discrepancia de datum 4326/4686 es despreciable frente al umbral  — máximo 0.000 mm
  PASA  El crudo de manzana coincide con su hash registrado
  PASA  Cada punto de los casos recibe exactamente una manzana  — 12 filas para 12 puntos
  PASA  La separación entre manzanas no se presenta como ancho de vía  — mediana 5.4 m, declarada como separación catastral y no como calle
  PASA  Todo veredicto adoptado es estable en la banda de tolerancia  — 2 adoptados de 6 sobre la banda [2.7, 6.4, 10.8, 21.5] m
  PASA  La regla de clasificación se reimplementó sin discrepancias  — 0 pares difieren de B1
  PASA  La mediana varía de

# Guardado

In [16]:
COLS_MZ = ["MANCODIGO", "SECCODIGO", "area_m2", "perim_m",
           "lado_corto_m", "lado_largo_m", "diagonal_m"]

MZ[COLS_MZ].to_csv(RUTA_OUT / "B2_dimensiones_manzana_ciudad.csv",
                   index=False, encoding="utf-8-sig")
MZ_LOCAL[COLS_MZ].to_csv(RUTA_OUT / "B2_dimensiones_manzana_local.csv",
                         index=False, encoding="utf-8-sig")
VD.to_csv(RUTA_OUT / "B2_verificacion_distancias.csv",
          index=False, encoding="utf-8-sig")
RES.to_csv(RUTA_OUT / "B2_contencion_manzana_casos.csv",
           index=False, encoding="utf-8-sig")
SENS.to_csv(RUTA_OUT / "B2_sensibilidad_umbrales.csv",
            index=False, encoding="utf-8-sig")
SENS_RADIO.to_csv(RUTA_OUT / "B2_sensibilidad_radio_entorno.csv",
                  index=False, encoding="utf-8-sig")
SENS_TOL.to_csv(RUTA_OUT / "B2_sensibilidad_tolerancia_veredicto.csv",
                encoding="utf-8-sig")
SEP.to_frame().to_csv(RUTA_OUT / "B2_separacion_entre_manzanas.csv",
                      index=False, encoding="utf-8-sig")
TESTS.to_csv(RUTA_OUT / "B2_pruebas_aceptacion.csv",
             index=False, encoding="utf-8-sig")

(RUTA_OUT / "B2_procedencia_manzana.json").write_text(
    json.dumps(procedencia, ensure_ascii=False, indent=2), encoding="utf-8")

# La fecha de verificación es la de la DESCARGA del insumo, no la de la
# corrida: dos corridas del mismo crudo producen el mismo archivo.
(RUTA_OUT / "B2_veredictos.json").write_text(json.dumps({
    "fecha_verificacion": FECHA_DESCARGA_MANZANA,
    "metodo": ("contención en manzana catastral UAECD (Mapa de Referencia "
               "IDECA), con tolerancia evaluada sobre una banda medida en la "
               "misma capa y criterio de suficiencia por área del polígono; "
               "solo se adoptan veredictos estables en toda la banda"),
    "fuente": procedencia,
    "tolerancia_adoptada_m": TOL_CONTENCION_M,
    "banda_tolerancia_m": BANDA_TOLERANCIA_M,
    "separacion_mediana_manzanas_m": round(SEPARACION_MEDIANA_MZ, 1),
    "separacion_p90_manzanas_m": round(SEPARACION_P90, 1),
    "nota_separacion": ("la separación entre manzanas de la capa catastral no "
                        "equivale al ancho de vía: se usa como referencia de "
                        "escala, no como medida de calle"),
    "area_max_manzana_m2": LIM_SUP_M2,
    "resueltos": int(RES.veredicto.notna().sum()), "total": len(RES),
    "veredictos": {r.caso: (None if pd.isna(r.veredicto) else bool(r.veredicto))
                   for _, r in RES.iterrows()},
    "motivos": {r.caso: r.motivo for _, r in RES.iterrows()},
}, ensure_ascii=False, indent=2), encoding="utf-8")

(RUTA_OUT / "B2_auditoria_procesamiento.json").write_text(json.dumps({
    "corte_inventario": constantes.FECHA_CORTE_LOCAL,
    "crs_geografico": CRS_GEO,
    "crs_metrico": CRS_PROY,
    "insumos_sha256": MANIFIESTO,
    "umbrales_evaluados": {
        "UMBRAL_CONFIRMADA": UMBRAL_CONFIRMADA,
        "UMBRAL_PROBABLE": UMBRAL_PROBABLE,
        "UMBRAL_TEXTUAL": UMBRAL_TEXTUAL,
        "RADIO_BUSQUEDA_MAX": RADIO_BUSQUEDA_MAX,
        "RADIO_SECUNDARIO": RADIO_SECUNDARIO,
        "RADIO_DUPLICADO": RADIO_DUPLICADO,
    },
    "parametros_b2": {
        "LIM_INF_M2": LIM_INF_M2, "LIM_SUP_M2": LIM_SUP_M2,
        "RADIO_LOCAL_M": RADIO_LOCAL_M, "MUESTRA_VIA": MUESTRA_VIA,
        "SEMILLA": SEMILLA,
        "CORTE_CONTIGUIDAD_M": CORTE_CONTIGUIDAD_M,
        "BANDA_TOLERANCIA_M": BANDA_TOLERANCIA_M,
        "TOL_CONTENCION_M": TOL_CONTENCION_M,
    },
    "escala_medida": {
        "manzanas_ciudad": int(len(MZ)),
        "manzanas_entorno": int(len(MZ_LOCAL)),
        "lado_corto_mediano_m": round(float(med_corto), 1),
        "lado_largo_mediano_m": round(float(med_largo), 1),
        "diagonal_mediana_m": round(float(med_diag), 1),
        "separacion_mediana_manzanas_m": round(SEPARACION_MEDIANA_MZ, 1),
        "separacion_no_contigua_mediana_m": round(SEPARACION_NO_CONTIGUA_MED, 1),
        "separacion_p90_manzanas_m": round(SEPARACION_P90, 1),
    },
    "sesgo_9377_ppm": {
        c: round(float(g.sesgo_ppm.mean()), 1)
        for c, g in VD.groupby("ciudad")
    },
    "pruebas": PRUEBAS,
    "limitacion_declarada": (
        "El anclaje de umbral se mide sobre la capa catastral de Bogotá; "
        "no hay equivalente abierto para Medellín, de modo que el umbral se "
        "transfiere sin verificación local. La separación entre manzanas de "
        "la capa no equivale al ancho de vía y no se usa como tal."
    ),
}, ensure_ascii=False, indent=2), encoding="utf-8")

print("Guardado en B2. Las salidas de B1 quedan intactas.")
print("Siguiente paso: B2.1.ipynb")

Guardado en B2. Las salidas de B1 quedan intactas.
Siguiente paso: B2.1.ipynb
